In [2]:
from datasets import load_dataset

ds = load_dataset("erdem-erdem/Turkish-Law-Documents-700k-clustered")

c:\Users\Aybars\miniconda3\envs\student_analysis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds.save_to_disk(r'C:\work environment\Python\nlp_exercises\saved_dataset')

Saving the dataset (9/9 shards): 100%|██████████| 702295/702295 [00:01<00:00, 396697.76 examples/s]


In [4]:
ds['train'][0]

{'text': '**2. Hukuk Dairesi \xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa02007/17004 E. \xa0, \xa02007/14843 K.**\n\n**"İçtihat Metni"**\n\nMAHKEMESİ :Fatih 2.Asliye Hukuk Mahkemesi\n\nTARİHİ :16.6.2006\n\nTaraflar arasındaki davanın yapılan muhakemesi sonunda mahalli mahkemece verilen ve yukarıda tarih numarası gösterilen hüküm temyiz edilmekle evrak okunup gereği görüşülüp düşünüldü.\n\nDavacı Hazine vekili dava dilekçesinde; 2230 ada 3 praselde kayıtlı taşınmazın 1/6 payının Arnavut tebalı A, 1/6 payının da yine Arnavut tebalı E adına kayıtlı olduğunu, bu kişilere ait hisselerin, kayyımla idaresine karar verildiğini, kayyımla idare süresinin 10 yılı doldurduğunu ileri sürerek, A ile E\'nın gaipliklerine, bu kişiler adına kayıtlı payların Hazine adına tesciline ve tapu kaydına konan şerhlerin de kaldırılmasına karar verilmesini istemiştir.\n\nTaşınmazın tapu kaydı üzerindeki şerh, 4.2.1992 tarih ve 323 yevmiye numarası ile konulan A ve E hisseleri için İstanbul Defterdarının kayyım tayin edildiği

In [5]:
len(ds['train'])

702295

In [6]:
sample = ds['train'].select(range(100))
df = sample.to_pandas()
df.head()

,text,source,tr-e5_knn_cluster_id,orig-e5_knn_cluster_id,tr-e5_hdbscan_cluster_id,orig-e5_hdbscan_cluster_id,id,esasNo,kararNo,kararTarihi
0,"**2. Hukuk Dairesi 2007/17004 E. , 2...",ygty1,24,3,-1,-1,80600600,2007/17004,2007/14843,02.11.2007
1,"**17. Ceza Dairesi 2016/12248 E. , 2...",ygty1,10,-1,11,16,361398900,2016/12248,2017/7610,13.06.2017
2,"**1. Ceza Dairesi 2021/7354 E. , 202...",ygty1,25,1,38,29,699062500,2021/7354,2021/13000,04.10.2021
3,"**7. Ceza Dairesi 2015/14826 E. , 20...",ygty1,25,1,42,26,197123900,2015/14826,2016/6225,27.04.2016
4,"**1. Hukuk Dairesi 2010/8760 E. , 20...",ygty1,24,3,55,-1,78540800,2010/8760,2010/13386,14.12.2010


In [31]:
import pandas as pd

def normalize_text_column(df, text_column='text', output_column='text_clean'):
    result = df.copy()
    cleaned = result[text_column].astype('string')
    cleaned = cleaned.str.replace('\xa0', ' ', regex=False)
    cleaned = cleaned.str.replace('**', '', regex=False)
    cleaned = cleaned.str.replace(r'\r\n|\r', '\n', regex=True)
    cleaned = cleaned.str.replace(r'[\t ]+', ' ', regex=True)
    cleaned = cleaned.str.replace(r'\n{3,}', '\n\n', regex=True)
    result[output_column] = cleaned.str.strip()
    return result

df_clean = normalize_text_column(df, text_column='text', output_column='text_clean')
df_clean[['text', 'text_clean']].head()

,text,text_clean
0,"**2. Hukuk Dairesi 2007/17004 E. , 2...","2. Hukuk Dairesi 2007/17004 E. , 2007/14843 K...."
1,"**17. Ceza Dairesi 2016/12248 E. , 2...","17. Ceza Dairesi 2016/12248 E. , 2017/7610 K.\..."
2,"**1. Ceza Dairesi 2021/7354 E. , 202...","1. Ceza Dairesi 2021/7354 E. , 2021/13000 K.\n..."
3,"**7. Ceza Dairesi 2015/14826 E. , 20...","7. Ceza Dairesi 2015/14826 E. , 2016/6225 K.\n..."
4,"**1. Hukuk Dairesi 2010/8760 E. , 20...","1. Hukuk Dairesi 2010/8760 E. , 2010/13386 K.\..."


In [32]:
import re
from collections import Counter

patterns = {
    'SONUÇ': re.compile(r'\bSONUÇ\s*:', re.IGNORECASE),
    'HÜKÜM': re.compile(r'\bHÜKÜM\s*:', re.IGNORECASE),
    'KARAR': re.compile(r'\bKARAR\s*:', re.IGNORECASE),
    'GEREĞİ DÜŞÜNÜLDÜ': re.compile(r'\bGEREĞİ\s+DÜŞÜNÜLDÜ\s*:', re.IGNORECASE),
}

def normalize_for_heading_search(text):
    text = text.replace('\xa0', ' ')
    text = text.replace('**', '')
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[\t ]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text

counts = Counter()
union = 0
any_two_plus = 0

for text in ds['train']['text']:
    normalized = normalize_for_heading_search(text)
    hits = [name for name, pattern in patterns.items() if pattern.search(normalized)]
    if hits:
        union += 1
        counts.update(hits)
        if len(hits) >= 2:
            any_two_plus += 1

total = len(ds['train'])
result = {
    'total_rows': total,
    'counts': {name: counts[name] for name in patterns},
    'rates': {name: counts[name] / total for name in patterns},
    'any_of_four_count': union,
    'any_of_four_rate': union / total,
    'two_or_more_count': any_two_plus,
    'two_or_more_rate': any_two_plus / total,
}

result

{'total_rows': 702295,
 'counts': {'SONUÇ': 205705,
  'HÜKÜM': 158402,
  'KARAR': 26461,
  'GEREĞİ DÜŞÜNÜLDÜ': 192154},
 'rates': {'SONUÇ': 0.2929039790971031,
  'HÜKÜM': 0.2255490926177746,
  'KARAR': 0.03767789888864366,
  'GEREĞİ DÜŞÜNÜLDÜ': 0.2736086687218334},
 'any_of_four_count': 457441,
 'any_of_four_rate': 0.6513516399803502,
 'two_or_more_count': 123089,
 'two_or_more_rate': 0.17526680383599486}

In [ ]:
import re
from collections import Counter
from random import Random

keyword_patterns = {
    'SONUÇ': re.compile(r'\bSONUÇ\s*:', re.IGNORECASE),
    'HÜKÜM': re.compile(r'\bHÜKÜM\s*:', re.IGNORECASE),
    'KARAR': re.compile(r'\bKARAR\s*:', re.IGNORECASE),
    'GEREĞİ DÜŞÜNÜLDÜ': re.compile(r'\bGEREĞİ\s+DÜŞÜNÜLDÜ\s*:', re.IGNORECASE),
}

def normalize_for_search(text):
    text = text.replace('\xa0', ' ')
    text = text.replace('**', '')
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[\t ]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text

def has_any_keyword(text):
    normalized = normalize_for_search(text)
    return any(pattern.search(normalized) for pattern in keyword_patterns.values())

stopwords = {
    've', 'veya', 'ile', 'bu', 'şu', 'o', 'bir', 'da', 'de', 'ki', 'mi', 'mı', 'mu', 'mü',
    'için', 'gibi', 'olan', 'olarak', 'çok', 'az', 'daha', 'en', 'ama', 'fakat', 'ise', 'dır',
    'dir', 'tır', 'tir', 'tur', 'tür', 'ile', 'tarafından', 'hakkında', 'sonra', 'önce', 'göre',
    'yapılan', 'edilen', 'olan', 'olup', 'olduğu', 'sayılı', 'maddesi', 'maddesine', 'mahkeme',
    'mahkemesi', 'dosya', 'karar', 'hüküm', 'sonuç', 'gereği', 'düşünüldü', 'temyiz', 'istinaf',
    'dava', 'davacı', 'davalı', 'sanık', 'sanığın', 'sanığın', 'müvekkil', 'vekil', 'dosyanın'
}

token_pattern = re.compile(r"[a-zA-ZçğıöşüÇĞİÖŞÜ]{3,}")

def tokenize(text):
    return [token.lower() for token in token_pattern.findall(text)]

no_keyword_texts = [text for text in ds['train']['text'] if not has_any_keyword(text)]
sample_size = 1000
rng = Random(42)
sample_indices = rng.sample(range(len(no_keyword_texts)), min(sample_size, len(no_keyword_texts)))
sample_texts = [no_keyword_texts[i] for i in sample_indices]

word_counts = Counter()
bigram_counts = Counter()
for text in sample_texts:
    normalized = normalize_for_search(text)
    tokens = [token for token in tokenize(normalized) if token not in stopwords]
    word_counts.update(tokens)
    bigram_counts.update(zip(tokens, tokens[1:]))

results_no_keyword = {
    'no_keyword_total': len(no_keyword_texts),
    'no_keyword_rate': len(no_keyword_texts) / len(ds['train']),
    'sample_size': len(sample_texts),
    'top_words': word_counts.most_common(40),
    'top_bigrams': bigram_counts.most_common(40),
    'examples': [normalize_for_search(t)[:500] for t in sample_texts[:5]],
}

results_no_keyword

{'no_keyword_total': 244854,
 'no_keyword_rate': 0.3486483600196499,
 'sample_size': 1000,
 'top_words': [('konusu', 3424),
  ('ilişkin', 2940),
  ('yer', 2617),
  ('tarih', 2438),
  ('kanun', 2056),
  ('verilen', 1961),
  ('hukuk', 1948),
  ('kararının', 1943),
  ('i̇dare', 1884),
  ('uygun', 1848),
  ('vergi', 1821),
  ('uyarınca', 1820),
  ('nun', 1743),
  ('kanunu', 1685),
  ('bölge', 1608),
  ('maddesinin', 1563),
  ('alan', 1550),
  ('kamu', 1509),
  ('davacının', 1462),
  ('kararı', 1435),
  ('maddesinde', 1422),
  ('davanın', 1408),
  ('tarihinde', 1398),
  ('ilgili', 1396),
  ('kararın', 1350),
  ('ceza', 1290),
  ('hukuka', 1257),
  ('üzere', 1188),
  ('özel', 1178),
  ('dairesi', 1167),
  ('bulunan', 1160),
  ('tarihli', 1148),
  ('nın', 1147),
  ('nin', 1140),
  ('gerektiği', 1133),
  ('şekilde', 1129),
  ('reddine', 1116),
  ('esas', 1091),
  ('yargılama', 1066),
  ('konu', 1030)],
 'top_bigrams': [(('kanunu', 'nun'), 1256),
  (('yer', 'alan'), 1063),
  (('bölge', 'i̇dare'

In [34]:
spaced_patterns = {
    'K A R A R': re.compile(r'K\s*A\s*R\s*A\s*R\s*[:：]?', re.IGNORECASE),
    'H Ü K Ü M': re.compile(r'H\s*Ü\s*K\s*Ü\s*M\s*[:：]?', re.IGNORECASE),
    'S O N U Ç': re.compile(r'S\s*O\s*N\s*U\s*Ç\s*[:：]?', re.IGNORECASE),
    'G E R E Ğ İ D Ü Ş Ü N Ü L D Ü': re.compile(r'G\s*E\s*R\s*E\s*Ğ\s*İ\s+D\s*Ü\s*Ş\s*Ü\s*N\s*Ü\s*L\s*D\s*Ü\s*[:：]?', re.IGNORECASE),
}

spaced_counts = Counter()
spaced_union = 0

for text in no_keyword_texts:
    normalized = normalize_for_search(text)
    hits = [name for name, pattern in spaced_patterns.items() if pattern.search(normalized)]
    if hits:
        spaced_union += 1
        spaced_counts.update(hits)

spaced_results = {
    'no_keyword_total': len(no_keyword_texts),
    'spaced_counts': {name: spaced_counts[name] for name in spaced_patterns},
    'spaced_rates': {name: spaced_counts[name] / len(no_keyword_texts) for name in spaced_patterns},
    'any_spaced_heading_count': spaced_union,
    'any_spaced_heading_rate': spaced_union / len(no_keyword_texts),
}

spaced_results

{'no_keyword_total': 244854,
 'spaced_counts': {'K A R A R': 241965,
  'H Ü K Ü M': 166996,
  'S O N U Ç': 49900,
  'G E R E Ğ İ D Ü Ş Ü N Ü L D Ü': 16720},
 'spaced_rates': {'K A R A R': 0.9882011321032125,
  'H Ü K Ü M': 0.6820227564181104,
  'S O N U Ç': 0.20379491452048976,
  'G E R E Ğ İ D Ü Ş Ü N Ü L D Ü': 0.06828559059684547},
 'any_spaced_heading_count': 241988,
 'any_spaced_heading_rate': 0.9882950656309474}